In [2]:

import torch
import pandas as pd
from pykeen.triples import TriplesFactory
from pykeen.models import DistMult, CompGCN, NodePiece
from pykeen.training import SLCWATrainingLoop
from pykeen.losses import MarginRankingLoss
from pykeen.evaluation import RankBasedEvaluator, SampledRankBasedEvaluator
from torch.optim import Adam, RMSprop, NAdam

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [3]:
main_data = pd.read_csv('../data/edges/triples.csv')
main_data = main_data.astype(str)

triples = main_data[['id_entity_1', 'predicate', 'id_entity_2']].values
triplet_data = TriplesFactory.from_labeled_triples(triples, create_inverse_triples=True)
training_set, testing_set, validation_set = triplet_data.split([0.8, 0.1, 0.1], random_state=17)


In [4]:
EMB_DIM = 128
LR = 1e-3
MARGIN = 1.2
WEIGHT = 1e-3
EPOCHS = 10
BATCH_SIZE = 4096
NUM_NEGS_PER_POS = 50

loss_function = MarginRankingLoss(margin=MARGIN)

model = DistMult(
    triples_factory=training_set,
    embedding_dim=EMB_DIM,
    random_seed=17,
    loss = loss_function,
)
model = model.to(device)


optimizer = Adam(params=model.parameters(), lr=LR)

training_loop = SLCWATrainingLoop(
    model=model,
    triples_factory=training_set,
    optimizer=optimizer,
    negative_sampler='bernoulli',
    negative_sampler_kwargs=dict(
        num_negs_per_pos=NUM_NEGS_PER_POS
    )
)

training_loop.train(
    num_epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    triples_factory=training_set,
    use_tqdm_batch=False,
)

evaluator = RankBasedEvaluator()

model_results = evaluator.evaluate(
    model=model,
    mapped_triples=testing_set.mapped_triples.to(device),
    additional_filter_triples=[
            training_set.mapped_triples.to(device),
            validation_set.mapped_triples.to(device),
        ],
)


metrics = model_results.to_df()
metrics = metrics[(metrics['Side'] == 'both') & (metrics['Rank_type'] == 'realistic')]
metrics

Training epochs on cuda:0: 100%|██████████| 10/10 [20:17<00:00, 121.76s/epoch, loss=1.2, prev_loss=1.2]
Evaluating on cuda:0:   0%|          | 16.0/506k [00:09<79:24:03, 1.77triple/s]


KeyboardInterrupt: 

In [5]:
evaluator = RankBasedEvaluator()

model_results = evaluator.evaluate(
    model=model,
    mapped_triples=testing_set.mapped_triples[:10000].to(device),
    additional_filter_triples=[
            training_set.mapped_triples.to(device),
            validation_set.mapped_triples.to(device),
        ],
)


metrics = model_results.to_df()
metrics = metrics[(metrics['Side'] == 'both') & (metrics['Rank_type'] == 'realistic')]
metrics

Evaluating on cuda:0: 100%|██████████| 10.0k/10.0k [01:55<00:00, 86.4triple/s]


,Side,Rank_type,Metric,Value
5,both,realistic,median_rank,5.350670e+05
14,both,realistic,z_geometric_mean_rank,-3.472636e-01
23,both,realistic,z_inverse_harmonic_mean_rank,-5.648999e-01
32,both,realistic,count,2.000000e+04
41,both,realistic,inverse_geometric_mean_rank,2.526122e-06
50,both,realistic,adjusted_inverse_harmonic_mean_rank,-4.944589e-06
59,both,realistic,median_absolute_deviation,3.966851e+05
68,both,realistic,adjusted_geometric_mean_rank_index,-2.455350e-03
77,both,realistic,variance,9.503565e+10
86,both,realistic,adjusted_arithmetic_mean_rank_index,1.656573e-03
